In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import openmeteo_requests
from datetime import datetime, timedelta

import pandas as pd
import requests_cache
from retry_requests import retry

import pvlib

import pickle

In [2]:
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

In [3]:
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": -29,
	"longitude": 24,
	"hourly": ["temperature_2m", "relative_humidity_2m", "precipitation", "snowfall", "cloud_cover", "cloud_cover_low", "cloud_cover_mid", "cloud_cover_high", "wind_speed_10m", "wind_speed_80m", "wind_direction_10m", "wind_direction_80m", "shortwave_radiation_instant", "wind_gusts_10m"],
	"current": ["precipitation", "snowfall", "cloud_cover", "wind_speed_10m", "wind_direction_10m", "temperature_2m", "relative_humidity_2m"],
	"timezone": "auto",
}
responses = openmeteo.weather_api(url, params = params)

In [4]:
response = responses[0]
current = response.Current()
hourly = response.Hourly()

In [5]:
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(2).ValuesAsNumpy()
hourly_snowfall = hourly.Variables(3).ValuesAsNumpy()
hourly_cloud_cover = hourly.Variables(4).ValuesAsNumpy()
hourly_cloud_cover_low = hourly.Variables(5).ValuesAsNumpy()
hourly_cloud_cover_mid = hourly.Variables(6).ValuesAsNumpy()
hourly_cloud_cover_high = hourly.Variables(7).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(8).ValuesAsNumpy()
hourly_wind_speed_80m = hourly.Variables(9).ValuesAsNumpy()
hourly_wind_direction_10m = hourly.Variables(10).ValuesAsNumpy()
hourly_wind_direction_80m = hourly.Variables(11).ValuesAsNumpy()
hourly_shortwave_radiation_instant = hourly.Variables(12).ValuesAsNumpy()
hourly_wind_gusts_10m = hourly.Variables(13).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	end =  pd.to_datetime(hourly.TimeEnd() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["precipitation"] = hourly_precipitation
hourly_data["snowfall"] = hourly_snowfall
hourly_data["cloud_cover"] = hourly_cloud_cover
hourly_data["cloud_cover_high"] = hourly_cloud_cover_high
hourly_data["cloud_cover_mid"] = hourly_cloud_cover_mid
hourly_data["cloud_cover_low"] = hourly_cloud_cover_low
hourly_data["shortwave_radiation_instant"] = hourly_shortwave_radiation_instant
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
hourly_data["wind_direction_10m"] = hourly_wind_direction_10m
hourly_data["wind_speed_80m"] = hourly_wind_speed_80m
hourly_data["wind_direction_80m"] = hourly_wind_direction_80m
hourly_data["wind_gusts_10m"] = hourly_wind_gusts_10m

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)


Hourly data
                          date  temperature_2m  relative_humidity_2m  \
0   2026-05-06 00:00:00+00:00            6.60             81.448975   
1   2026-05-06 01:00:00+00:00            6.25             83.141815   
2   2026-05-06 02:00:00+00:00            5.90             85.174873   
3   2026-05-06 03:00:00+00:00            5.60             86.961739   
4   2026-05-06 04:00:00+00:00            5.30             88.478409   
..                        ...             ...                   ...   
163 2026-05-12 19:00:00+00:00           13.95             76.064735   
164 2026-05-12 20:00:00+00:00           13.30             78.818451   
165 2026-05-12 21:00:00+00:00           12.70             81.149040   
166 2026-05-12 22:00:00+00:00           12.05             82.996811   
167 2026-05-12 23:00:00+00:00           11.35             85.182487   

     precipitation  snowfall  cloud_cover  cloud_cover_high  cloud_cover_mid  \
0              0.0       0.0          0.0            

In [6]:
now = datetime.now().replace(microsecond=0, second=0, minute=0)
end = pd.Timestamp(now + timedelta(days=1), tz='UTC')
now = pd.Timestamp(now, tz='UTC')

In [7]:
data = hourly_dataframe[hourly_dataframe["date"] >= now]
data = data[data["date"] <= end]

In [8]:
az = pvlib.solarposition.get_solarposition(data["date"], params["latitude"], params["longitude"]).drop(["apparent_zenith", "apparent_elevation", "equation_of_time", "elevation"], axis = 1)

In [9]:
data["angle_of_incidence"] = abs(params["latitude"])
data["zenith"] = np.array(az['zenith'])
data["azimuth"] = np.array(az["azimuth"])
time = np.array(data["date"])
data = data.drop("date", axis=1)

In [10]:
data

,temperature_2m,relative_humidity_2m,precipitation,snowfall,cloud_cover,cloud_cover_high,cloud_cover_mid,cloud_cover_low,shortwave_radiation_instant,wind_speed_10m,wind_direction_10m,wind_speed_80m,wind_direction_80m,wind_gusts_10m,angle_of_incidence,zenith,azimuth
16,7.80,88.388527,1.0,0.0,100.0,84.0,100.0,100.0,47.428017,29.452425,278.080475,48.812866,283.216339,47.880001,29,93.665210,287.042582
17,8.15,83.954643,0.4,0.0,100.0,98.0,100.0,100.0,16.223114,33.085255,280.025970,44.793606,283.477844,50.399998,29,106.413385,280.299533
18,8.95,78.980881,0.2,0.0,100.0,24.0,100.0,100.0,0.000000,31.862036,269.352631,47.912468,277.338501,51.119999,29,119.432763,273.684413
19,9.50,79.885094,0.1,0.0,100.0,4.0,100.0,100.0,0.000000,29.070974,262.171021,50.457821,277.378448,48.239998,29,132.548289,266.400191
20,10.10,79.151176,0.0,0.0,100.0,1.0,100.0,100.0,0.000000,29.987158,262.064545,47.194340,272.185760,45.360001,29,145.528705,256.910267
21,10.40,79.195274,0.0,0.0,100.0,0.0,74.0,100.0,0.000000,32.352962,256.814240,40.323215,262.304047,48.959999,29,157.805966,240.810290
22,10.45,79.746925,0.0,0.0,77.0,0.0,68.0,44.0,0.000000,32.402500,249.853653,38.268520,258.055847,50.759998,29,166.857509,202.174397
23,10.10,82.192635,0.0,0.0,39.0,0.0,30.0,27.0,0.000000,28.233044,251.795990,35.900864,254.291290,49.680000,29,164.764150,141.397843
24,9.70,84.423607,0.0,0.0,30.0,0.0,6.0,27.0,0.000000,24.107906,246.689041,32.744175,255.352585,42.839996,29,154.114377,112.776902
25,8.95,87.584068,0.0,0.0,18.0,0.0,8.0,13.0,0.000000,21.237711,250.182755,30.853069,258.558990,36.719997,29,141.509161,99.615755


In [11]:
nkeys = pd.read_csv("normalization_keys.csv")

In [12]:
ykey = nkeys["generated_power_kw"]
nkeys = nkeys.drop("generated_power_kw", axis = 1)

In [13]:
def normalize(df, nkeys):
    df_new = df.copy()
    for key in df_new.keys():
        x = df[key]
        a = nkeys[key][0]
        b = nkeys[key][1]
        df_new[key] = (x-a)/(b-a)
    return df_new

def denormalize_array(arr, normalization_key):
    a = normalization_key[0]
    b = normalization_key[1]
    return (b-a)*arr+a

def denormalize(df, normalization_keys, column = None):
    df_new = df.copy()
    if column:
        df_new = denormalize_array(df_new, normalization_keys[column])
    else:
        for key in df_new.keys():
            df_new[key] = denormalize_array(df[key], normalization_keys[key])
    return df_new

In [14]:
normalized_data = normalize(data, nkeys)

In [15]:
normalized_data.keys()

Index(['temperature_2m', 'relative_humidity_2m', 'precipitation', 'snowfall',
       'cloud_cover', 'cloud_cover_high', 'cloud_cover_mid', 'cloud_cover_low',
       'shortwave_radiation_instant', 'wind_speed_10m', 'wind_direction_10m',
       'wind_speed_80m', 'wind_direction_80m', 'wind_gusts_10m',
       'angle_of_incidence', 'zenith', 'azimuth'],
      dtype='str')

In [16]:
with open('model.pkl', 'rb') as f:
    model = pickle.load(f)

In [17]:
model

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",75
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",3
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",0.5
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples a

In [18]:
prediction = model.predict(data)

In [19]:
result = denormalize_array(prediction, ykey)

In [20]:
time

array([Timestamp('2026-05-06 16:00:00+0000', tz='UTC'),
       Timestamp('2026-05-06 17:00:00+0000', tz='UTC'),
       Timestamp('2026-05-06 18:00:00+0000', tz='UTC'),
       Timestamp('2026-05-06 19:00:00+0000', tz='UTC'),
       Timestamp('2026-05-06 20:00:00+0000', tz='UTC'),
       Timestamp('2026-05-06 21:00:00+0000', tz='UTC'),
       Timestamp('2026-05-06 22:00:00+0000', tz='UTC'),
       Timestamp('2026-05-06 23:00:00+0000', tz='UTC'),
       Timestamp('2026-05-07 00:00:00+0000', tz='UTC'),
       Timestamp('2026-05-07 01:00:00+0000', tz='UTC'),
       Timestamp('2026-05-07 02:00:00+0000', tz='UTC'),
       Timestamp('2026-05-07 03:00:00+0000', tz='UTC'),
       Timestamp('2026-05-07 04:00:00+0000', tz='UTC'),
       Timestamp('2026-05-07 05:00:00+0000', tz='UTC'),
       Timestamp('2026-05-07 06:00:00+0000', tz='UTC'),
       Timestamp('2026-05-07 07:00:00+0000', tz='UTC'),
       Timestamp('2026-05-07 08:00:00+0000', tz='UTC'),
       Timestamp('2026-05-07 09:00:00+0000', tz=

In [21]:
for i in range(len(result)):
    print(time[i], result[i])

2026-05-06 16:00:00+00:00 432.2200834745642
2026-05-06 17:00:00+00:00 432.2200834745642
2026-05-06 18:00:00+00:00 119.74891666806573
2026-05-06 19:00:00+00:00 119.74891666806573
2026-05-06 20:00:00+00:00 119.91406649917687
2026-05-06 21:00:00+00:00 118.5114199754635
2026-05-06 22:00:00+00:00 118.5114199754635
2026-05-06 23:00:00+00:00 118.5114199754635
2026-05-07 00:00:00+00:00 118.5114199754635
2026-05-07 01:00:00+00:00 118.5114199754635
2026-05-07 02:00:00+00:00 118.5114199754635
2026-05-07 03:00:00+00:00 123.93770452278436
2026-05-07 04:00:00+00:00 123.93770452278436
2026-05-07 05:00:00+00:00 123.93770452278436
2026-05-07 06:00:00+00:00 123.93770452278436
2026-05-07 07:00:00+00:00 123.93770452278436
2026-05-07 08:00:00+00:00 435.837639104726
2026-05-07 09:00:00+00:00 435.837639104726
2026-05-07 10:00:00+00:00 435.837639104726
2026-05-07 11:00:00+00:00 435.837639104726
2026-05-07 12:00:00+00:00 435.837639104726
2026-05-07 13:00:00+00:00 435.837639104726
2026-05-07 14:00:00+00:00 431.

In [22]:
params

{'latitude': -29,
 'longitude': 24,
 'hourly': ['temperature_2m',
  'relative_humidity_2m',
  'precipitation',
  'snowfall',
  'cloud_cover',
  'cloud_cover_low',
  'cloud_cover_mid',
  'cloud_cover_high',
  'wind_speed_10m',
  'wind_speed_80m',
  'wind_direction_10m',
  'wind_direction_80m',
  'shortwave_radiation_instant',
  'wind_gusts_10m'],
 'current': ['precipitation',
  'snowfall',
  'cloud_cover',
  'wind_speed_10m',
  'wind_direction_10m',
  'temperature_2m',
  'relative_humidity_2m'],
 'timezone': 'auto'}